# AI Bootcamp — Section 4 · Practical 2: **Agent as Tool**
### Ollama + Qwen3 1.7B · Windows / JupyterLab · No AWS

**आपने Section 4 Practical 1 (Supervisor–Worker) पूरा कर लिया है।** अब एक Teacher Agent तीन अलग specialist agents को *tools* की तरह इस्तेमाल करेगा:
- **Math Tutor** — गणित को समझाता है
- **Writing Tutor** — भाषा और लेखन सुधारता है
- **Science Tutor** — विज्ञान समझाता है

हर tutor का अपना `system_prompt` है, लेकिन तीनों **एक ही स्थानीय Qwen3 model** इस्तेमाल करते हैं।

**क्या नया सीखेंगे?** `SpecialistAgent.as_tool()` किसी agent को ऐसे Python callable में बदलता है जिसे Teacher बुला सकता है। यह *Python-controlled tool orchestration* है: छोटा Qwen3 मॉडल खुद tools चुनने की जिम्मेदारी नहीं निभाता।

**Run:** Code Cell चुनें → `Shift + Enter` → अगले Code Cell पर जाएँ। **पहले Cell 1 चलाएँ।**

> यह शैक्षणिक अभ्यास है; कोई AWS, cloud API या live web search नहीं होती। तैयार जवाबों को पढ़कर जाँचें, क्योंकि छोटा मॉडल गलती कर सकता है।


## Cell 1 — Ollama और Qwen3 का Connection Check
अगर `Connection refused` आए तो Ollama App चालू करें। अगर Model नहीं मिले, PowerShell में `ollama pull qwen3:1.7b` चलाएँ।


In [ ]:
import json
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"

try:
    with urlopen(f"{OLLAMA_URL}/api/tags", timeout=15) as resp:
        installed_models = [m.get("name", "") for m in json.load(resp).get("models", [])]
except (URLError, TimeoutError, OSError) as exc:
    raise RuntimeError("Ollama server नहीं मिल रहा। Windows में Ollama App खोलें।") from exc

print("Installed models:", installed_models)
if MODEL not in installed_models:
    raise RuntimeError(f"{MODEL} नहीं मिला। नई PowerShell में: ollama pull {MODEL}")
print("PASS: Local Ollama + Qwen3 connected; AWS not required")


## Cell 2 — Local AI से बात करने वाला Function
यह केवल आपके कंप्यूटर के `127.0.0.1:11434/api/chat` को Call करता है। कोई Extra Python Package नहीं चाहिए।


In [ ]:
def ollama_chat(system_prompt, user_prompt, max_tokens=180):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "think": False,
        "keep_alive": "10m",
        "options": {
            "temperature": 0.2,
            "num_predict": max_tokens,
            "num_ctx": 2048,
        },
    }
    req = Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(req, timeout=180) as resp:
            data = json.load(resp)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:500]
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc
    except (URLError, TimeoutError, OSError) as exc:
        raise RuntimeError("Ollama से संपर्क नहीं हो पाया। Ollama App चालू है क्या?") from exc

    answer = data.get("message", {}).get("content", "").strip()
    if not answer:
        raise RuntimeError("Ollama ने खाली जवाब दिया; वही Cell दोबारा चलाएँ।")
    return answer

print("PASS: Local Ollama chat function ready")


## Cell 3 — तीन विशेषज्ञ AI Agents बनाएँ
`as_tool()` method की मदद से हर Specialist Agent को Python function-tool की तरह उपयोग कर सकते हैं।


In [ ]:
class SpecialistAgent:
    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt

    def as_tool(self):
        """Return a callable which runs this specialist via local Qwen3."""
        agent = self

        def call_specialist(question, verified_fact=""):
            details = f"Question: {question}"
            if verified_fact:
                details += f"\nPython-verified fact (do not change): {verified_fact}"
            result = ollama_chat(agent.system_prompt, details, max_tokens=185)
            return {"agent": agent.name, "question": question, "answer": result}

        call_specialist.__name__ = agent.name
        return call_specialist


math_agent = SpecialistAgent(
    "math_tutor",
    "You are a patient Hindi mathematics tutor for Class 5. Explain in 2-4 short "
    "Hindi sentences. Prefer the Python-verified fact given in the prompt. "
    "Do not invent formulas or arithmetic results.",
)
writing_agent = SpecialistAgent(
    "writing_tutor",
    "You are a Hindi writing teacher. Correct the provided sentence and explain "
    "one simple grammar or clarity improvement in Hindi. Keep the answer brief.",
)
science_agent = SpecialistAgent(
    "science_tutor",
    "You are a Hindi science teacher for Class 5. Explain the requested concept "
    "in 3 simple Hindi sentences. Do not claim to have browsed the web.",
)

specialist_tools = {
    "math": math_agent.as_tool(),
    "writing": writing_agent.as_tool(),
    "science": science_agent.as_tool(),
}
assert set(specialist_tools) == {"math", "writing", "science"}
print("PASS: 3 specialist AI agents exposed as callable tools")
print("Available tools:", list(specialist_tools))


## Cell 4 — Teacher Agent बनाकर Tools Connect करें
Teacher अपनी `ask_tool()` method से specialist को बुलाएगा और `combine()` में उनके जवाब को एक अध्ययन रिपोर्ट में जोड़ेगा।


In [ ]:
class TeacherAgent:
    def __init__(self, tools):
        self.tools = tools
        self.history = []

    def ask_tool(self, subject, question, verified_fact=""):
        if subject not in self.tools:
            raise ValueError(f"Unknown subject: {subject}; available: {list(self.tools)}")
        print(f"Teacher -> {subject} specialist")
        result = self.tools[subject](question, verified_fact=verified_fact)
        self.history.append(result)
        print(f"PASS: {result['agent']} responded")
        return result

    def combine(self):
        if len(self.history) < 3:
            raise RuntimeError("First run Cell 5 and Cell 6 for all 3 specialists.")
        sources = "\n\n".join(
            f"[{item['agent']}]\n{item['answer']}" for item in self.history[-3:]
        )
        return ollama_chat(
            "You are a Hindi primary school teacher. Summarize 3 specialist tutor answers "
            "into a concise learning recap. Stick to their supplied information; "
            "say it is a study exercise. Use short numbered points.",
            "These are 3 local specialist responses:\n" + sources,
            max_tokens=280,
        )


teacher = TeacherAgent(specialist_tools)
assert len(teacher.tools) == 3
print("PASS: Teacher Agent constructed with 3 specialist tools")


## Cell 5 — पहला Real Tool Call: Math Tutor
Teacher `math` tool को बुलाएगा। गणित का परिणाम पहले Python में निकालेंगे, ताकि AI को सत्यापित जवाब मिले।


In [ ]:
percent, number = 25, 240
verified_answer = number * percent / 100
math_result = teacher.ask_tool(
    "math",
    "240 का 25 प्रतिशत कितना है? इसे कक्षा 5 के विद्यार्थी को हिंदी में समझाओ।",
    verified_fact=f"25% of 240 = {verified_answer:g}",
)
print("\n--- MATH TUTOR ---\n")
print(math_result["answer"])
print("\nPASS: Teacher called the Math Agent as a tool")


## Cell 6 — Writing और Science Tutor को बुलाएँ
दो और Real Qwen3 calls होंगी। छोटे मॉडल पर CPU के अनुसार धीमा चल सकता है।


In [ ]:
writing_result = teacher.ask_tool(
    "writing",
    "इस वाक्य को सुधारो: 'मैं कल स्कूल गया था और किताब पढ़ता है।'",
)
print("\n--- WRITING TUTOR ---\n", writing_result["answer"])

science_result = teacher.ask_tool(
    "science",
    "प्रकाश संश्लेषण (photosynthesis) सरल हिंदी में समझाओ।",
)
print("\n--- SCIENCE TUTOR ---\n", science_result["answer"])

assert len(teacher.history) == 3
assert [i["agent"] for i in teacher.history] == [
    "math_tutor", "writing_tutor", "science_tutor"
]
print("\nPASS: All 3 specialist agents called via Teacher tools")


## Cell 7 — Teacher से Final Hindi Study Report बनाएँ
Teacher तीनों specialist outputs को पढ़कर एक छोटा Final Recap बनाएगा (1 और Real Qwen3 call)।


In [ ]:
final_study_report = teacher.combine()
print("========== FINAL TEACHER REPORT ==========\n")
print(final_study_report)
print("\nPASS: Teacher synthesized 3 specialist tool outputs")


## Cell 8 — रिपोर्ट Save करें
सफलता मिलने पर JupyterLab की बाईं File List में `.md` और `.json` रिपोर्ट मिलेंगी।


In [ ]:
assert final_study_report.strip(), "Run Cell 7 first"
md_file = Path("section4_agent_as_tool_report.md")
json_file = Path("section4_agent_as_tool_tool_calls.json")

md_file.write_text(
    "# AI Bootcamp Section 4 — Agent as Tool (Ollama/Qwen3)\n\n"
    "Local educational exercise. No AWS, paid API, or live web.\n\n"
    "## Final Teacher Report\n\n" + final_study_report + "\n",
    encoding="utf-8",
)
json_file.write_text(
    json.dumps(teacher.history, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("PASS: Section 4 Practical 2 — Agent as Tool completed")
print("Saved:", md_file.resolve())
print("Saved:", json_file.resolve())


---
## Troubleshooting (समस्या हो तो)
- **Cell 1 `Connection refused`:** Ollama Windows App खोलें। यदि Server पहले से नहीं चला है, दूसरी PowerShell में `ollama serve` चलाएँ।
- **Cell 1 `MODEL NOT FOUND`:** PowerShell में `ollama pull qwen3:1.7b` चलाएँ और फिर Cell 1 दोबारा Run करें।
- **`NameError`:** उस Cell के पहले वाले Code Cells ऊपर से नीचे क्रम में Run करें। यदि Kernel restart किया तो Cell 1 से दोबारा शुरू करें।
- **`[*]`:** Kernel busy है; Qwen3 लोकल CPU पर जवाब तैयार कर रहा है। एक साथ सभी Cells न चलाएँ।
- **गलत जवाब:** Qwen3 1.7B छोटा मॉडल है। यह अभ्यास architecture सिखाता है; तथ्यों को verify करें।
- **रिपोर्ट कहाँ मिली?** JupyterLab File Browser में current directory देखें, या Cell 8 में Saved path पढ़ें।

**अगला प्रैक्टिकल:** Section 4 — Graph Pattern (निर्धारित steps और branching workflow)।
